## 0 — Inter-Rater Reliability (unweighted)

Cohen's $\kappa$ between annotator 1 (the original human ground-truth labels,
`gt`) and annotator 2 (the revision re-annotation) on the shared sample of the
Sep 29 inspection set.

> This notebook is a two-annotator, sample-level cross-check. The canonical
> source for the paper's reported IRR is
> `0_interrater_agreement.ipynb` (three human annotators; Fleiss' $\kappa$ = 0.88).

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from sklearn.metrics import cohen_kappa_score, accuracy_score

BASE_DIR = Path("../../")
ANNOTATOR1_PATH = BASE_DIR / "data" / "revisions" / "irr" / "bayflood_annotator1.csv"
IRR_PATH = BASE_DIR / "data" / "revisions" / "irr" / "bayflood_annotator2.csv"


def frame_id(path_str):
    """Extract bare filename (no extension) from Label-Studio-style paths."""
    s = str(path_str)
    if "?d=" in s:
        s = s.split("?d=", 1)[1]
    return Path(s).stem

In [ ]:
# Annotator 1 HUMAN ground-truth labels (`gt`) — the comparison target.
annotator1 = pd.read_csv(ANNOTATOR1_PATH)
annotator1["frame_id"] = annotator1["image"].map(frame_id)
annotator1["gt"] = annotator1["gt"].astype(int)

# Load IRR annotations (Annotator 2 labels)
irr = pd.read_csv(IRR_PATH)
irr = irr.iloc[100:].copy()  # skip first 100 (training batch)
irr["choice"] = irr["choice"].astype(str).str.strip()
irr = irr[irr["choice"].isin(["Flooded", "Not Flooded"])]
irr["label_irr"] = irr["choice"].map({"Flooded": 1, "Not Flooded": 0})
irr["frame_id"] = irr["image"].map(frame_id)

# Merge: annotator-2 label + annotator-1 human gt
merged = pd.merge(
    irr[["frame_id", "label_irr"]],
    annotator1[["frame_id", "gt"]],
    on="frame_id",
    how="inner",
)
merged = merged.dropna(subset=["gt", "label_irr"])
merged["gt"] = merged["gt"].astype(int)
merged["label_irr"] = merged["label_irr"].astype(int)

print(f"IRR sample: {len(merged)} images")
print(f"\nAnnotator 1 (human gt) distribution:")
print(merged["gt"].value_counts(dropna=False).to_string())

In [ ]:
# Unweighted (sample-level) IRR — matches 09_interrater_agreement
kappa_uw = cohen_kappa_score(merged["label_irr"], merged["gt"])
acc_uw = accuracy_score(merged["label_irr"], merged["gt"])

ct_uw = pd.crosstab(
    merged["label_irr"], merged["gt"],
    rownames=["Annotator 2 (IRR)"], colnames=["Annotator 1 (human gt)"],
)

p_a1_pos_a2_pos_uw = ct_uw.loc[1, 1] / ct_uw.loc[1].sum()
p_a1_neg_a2_neg_uw = ct_uw.loc[0, 0] / ct_uw.loc[0].sum()

print("=== Unweighted IRR (sample) ===")
print(f"N = {len(merged)}")
print(f"Cohen's Kappa: {kappa_uw:.4f}")
print(f"Accuracy:      {acc_uw:.4f}")
print(f"\nP(A1=Flooded | A2=Flooded):         {p_a1_pos_a2_pos_uw:.4f}")
print(f"P(A1=Not Flooded | A2=Not Flooded): {p_a1_neg_a2_neg_uw:.4f}")
print(f"\nConfusion matrix:\n{ct_uw}")